<a href="https://colab.research.google.com/github/JudithJacquet/TFG-Pronostico-Picos-SIN/blob/prueba_modelos/modelo_24h_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo 24h para demanda del SIN

Este notebook predice las 24 horas del dia objetivo con regresion Ridge y luego obtiene el pico como el maximo de las 24 predicciones.


## 1. Funciones auxiliares
Ejecuta esta celda primero.


In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import files


def upload_dataset():
    print("Subi el archivo complete-dataset.csv cuando Colab te lo pida.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No se subio ningun archivo.")
    name = next(iter(uploaded))
    print(f"Archivo recibido: {name}")
    return pd.read_csv(io.BytesIO(uploaded[name]))


def find_datetime_column(df):
    if "DATETIME" in df.columns:
        return "DATETIME"
    return df.columns[0]


def find_demand_column(df):
    for candidate in ["SIN", "SIN Imputed", "sin", "demand"]:
        if candidate in df.columns:
            return candidate
    raise ValueError(f"No encontre columna de demanda. Columnas: {list(df.columns)}")


def prepare_hourly(df):
    dt_col = find_datetime_column(df)
    demand_col = find_demand_column(df)
    df = df.rename(columns={dt_col: "DATETIME_RAW", demand_col: "SIN"}).copy()

    # El dataset trae hora local con offset. Para agrupar por dia/hora local,
    # tomamos YYYY-MM-DD HH:MM:SS.
    dt_text = df["DATETIME_RAW"].astype(str).str.slice(0, 19)
    df["DATETIME"] = pd.to_datetime(dt_text, errors="coerce")
    df = df.dropna(subset=["DATETIME", "SIN"]).sort_values("DATETIME")
    df["date"] = df["DATETIME"].dt.normalize()
    df["hour"] = df["DATETIME"].dt.hour
    return df


def build_dataset(hourly):
    weather_cols = [c for c in ["T02M", "RH2M", "PRSS", "TPP6", "U10M", "V10M"] if c in hourly.columns]

    hourly_matrix = (
        hourly.pivot_table(index="date", columns="hour", values="SIN", aggfunc="mean")
        .sort_index()
        .reindex(columns=range(24))
        .dropna()
    )
    hourly_matrix.columns = [f"sin_h{h:02d}" for h in range(24)]

    daily = pd.DataFrame(index=hourly_matrix.index)
    daily["peak_real"] = hourly_matrix.max(axis=1)
    daily["peak_hour_real"] = hourly_matrix.to_numpy().argmax(axis=1)
    daily["sin_mean"] = hourly_matrix.mean(axis=1)
    daily["sin_min"] = hourly_matrix.min(axis=1)
    daily["sin_std"] = hourly_matrix.std(axis=1)

    daily["year"] = daily.index.year
    daily["month"] = daily.index.month
    daily["dayofweek"] = daily.index.dayofweek
    daily["is_weekend"] = (daily["dayofweek"] >= 5).astype(int)
    daily["month_sin"] = np.sin(2 * np.pi * daily["month"] / 12)
    daily["month_cos"] = np.cos(2 * np.pi * daily["month"] / 12)
    daily["dow_sin"] = np.sin(2 * np.pi * daily["dayofweek"] / 7)
    daily["dow_cos"] = np.cos(2 * np.pi * daily["dayofweek"] / 7)

    # En un escenario day-ahead, estas variables representan pronostico meteorologico
    # disponible para el dia objetivo.
    for col in weather_cols:
        grouped = hourly.groupby("date")[col].agg(["mean", "max", "min"])
        grouped.columns = [f"{col.lower()}_{stat}" for stat in grouped.columns]
        daily = daily.join(grouped, how="left")

    for lag in [1, 2, 3, 7, 14, 21, 28]:
        daily[f"peak_lag_{lag}"] = daily["peak_real"].shift(lag)
        daily[f"mean_lag_{lag}"] = daily["sin_mean"].shift(lag)

    for window in [3, 7, 14, 28]:
        daily[f"peak_roll_{window}_mean"] = daily["peak_real"].shift(1).rolling(window).mean()
        daily[f"peak_roll_{window}_std"] = daily["peak_real"].shift(1).rolling(window).std()

    # Perfil horario del dia anterior. Se usa en ambos enfoques para que la comparacion sea justa.
    for h in range(24):
        daily[f"prev_h{h:02d}"] = hourly_matrix[f"sin_h{h:02d}"].shift(1)

    data = daily.join(hourly_matrix, how="inner").dropna().reset_index(names="date")
    y24_cols = [f"sin_h{h:02d}" for h in range(24)]
    y24 = data[y24_cols].to_numpy(float)
    y_peak = data["peak_real"].to_numpy(float)

    excluded = {"date", "peak_real", "peak_hour_real", "sin_mean", "sin_min", "sin_std", *y24_cols}
    feature_cols = [c for c in data.columns if c not in excluded and pd.api.types.is_numeric_dtype(data[c])]
    return data, y24, y_peak, feature_cols


def chronological_split(data):
    years = sorted(data["date"].dt.year.unique())
    test_year = years[-1]
    val_year = years[-2]
    train_idx = data.index[data["date"].dt.year < val_year].to_numpy()
    val_idx = data.index[data["date"].dt.year == val_year].to_numpy()
    test_idx = data.index[data["date"].dt.year == test_year].to_numpy()
    return train_idx, val_idx, test_idx


def standardize(train_x, *arrays):
    mean = train_x.mean(axis=0)
    std = train_x.std(axis=0)
    std[std == 0] = 1
    return tuple((arr - mean) / std for arr in arrays)


def add_intercept(x):
    return np.column_stack([np.ones(len(x)), x])


def fit_ridge(x, y, alpha=10.0):
    penalty = alpha * np.eye(x.shape[1])
    penalty[0, 0] = 0
    return np.linalg.solve(x.T @ x + penalty, x.T @ y)


def peak_metrics(y_true, y_pred):
    error = y_pred - y_true
    abs_error = np.abs(error)
    under = y_pred < y_true
    under_amount = np.where(under, y_true - y_pred, np.nan)
    return {
        "peak_MAE_MW": float(abs_error.mean()),
        "peak_RMSE_MW": float(np.sqrt(np.mean(error**2))),
        "peak_MAPE_percent": float(np.mean(abs_error / y_true) * 100),
        "peak_bias_MW": float(error.mean()),
        "underestimation_rate_percent": float(under.mean() * 100),
        "mean_underestimation_MW": float(np.nanmean(under_amount)) if under.any() else 0.0,
        "p95_abs_peak_error_MW": float(np.percentile(abs_error, 95)),
    }


def hourly_metrics(y_true, y_pred):
    error = y_pred - y_true
    abs_error = np.abs(error)
    return {
        "hourly_MAE_MW": float(abs_error.mean()),
        "hourly_RMSE_MW": float(np.sqrt(np.mean(error**2))),
        "hourly_MAPE_percent": float(np.mean(abs_error / y_true) * 100),
    }


## 2. Carga del dataset
Cuando se ejecute, sube `complete-dataset.csv`.


In [ ]:
# Cargar datos
raw_df = upload_dataset()
hourly = prepare_hourly(raw_df)
data, y24, y_peak, feature_cols = build_dataset(hourly)
train_idx, val_idx, test_idx = chronological_split(data)

print("Dias validos:", len(data))
print("Train:", data.loc[train_idx, "date"].min().date(), "a", data.loc[train_idx, "date"].max().date(), len(train_idx))
print("Validation:", data.loc[val_idx, "date"].min().date(), "a", data.loc[val_idx, "date"].max().date(), len(val_idx))
print("Test:", data.loc[test_idx, "date"].min().date(), "a", data.loc[test_idx, "date"].max().date(), len(test_idx))
print("Features:", len(feature_cols))


## 3. Entrenamiento y metricas


In [ ]:
# Entrenamiento del modelo 24h
alpha = 10.0

x_raw = data[feature_cols].to_numpy(float)
train_x_s, val_x_s, test_x_s = standardize(x_raw[train_idx], x_raw[train_idx], x_raw[val_idx], x_raw[test_idx])
train_x = add_intercept(train_x_s)
val_x = add_intercept(val_x_s)
test_x = add_intercept(test_x_s)

weights_24h = fit_ridge(train_x, y24[train_idx], alpha=alpha)
pred_val_24h = val_x @ weights_24h
pred_test_24h = test_x @ weights_24h

pred_val_peak = pred_val_24h.max(axis=1)
pred_test_peak = pred_test_24h.max(axis=1)

metrics_24h = pd.DataFrame([
    {"split": "validation", **peak_metrics(y_peak[val_idx], pred_val_peak), **hourly_metrics(y24[val_idx], pred_val_24h)},
    {"split": "test", **peak_metrics(y_peak[test_idx], pred_test_peak), **hourly_metrics(y24[test_idx], pred_test_24h)},
])

metrics_24h.round(3)


## 4. Guardar resultados
Esta celda genera CSV y los descarga desde Colab.


In [ ]:
# Guardar y descargar resultados
test_dates = data.loc[test_idx, "date"].reset_index(drop=True)

peak_predictions = pd.DataFrame({
    "date": test_dates,
    "peak_real": y_peak[test_idx],
    "peak_hour_real": data.loc[test_idx, "peak_hour_real"].to_numpy(),
    "peak_pred_from_24h": pred_test_peak,
    "peak_error": pred_test_peak - y_peak[test_idx],
    "underestimated": pred_test_peak < y_peak[test_idx],
})

hourly_pred = pd.DataFrame(pred_test_24h, columns=[f"pred_h{h:02d}" for h in range(24)])
hourly_real = pd.DataFrame(y24[test_idx], columns=[f"real_h{h:02d}" for h in range(24)])
hourly_predictions = pd.concat([pd.DataFrame({"date": test_dates}), hourly_real, hourly_pred], axis=1)

metrics_24h.to_csv("metrics_24h.csv", index=False)
peak_predictions.to_csv("peak_predictions_from_24h.csv", index=False)
hourly_predictions.to_csv("hourly_predictions_24h.csv", index=False)

files.download("metrics_24h.csv")
files.download("peak_predictions_from_24h.csv")
files.download("hourly_predictions_24h.csv")
